In [ ]:
import re

SYSTEM_PROMPT = """
Ты профессионал по работе с клиентскими обращениями. Запомни метрики и их классификацию в рамках клиентского сервиса.
'101': 'Финансовые потери клиентов по вине Банка / компании Группы Выявлены отклонения и проблемы в клиентском сервисе, связанные с недостатками процессов Банка и повлекшие возникновение финансовых потерь'
'102': 'Необоснованный отказ в оказании услуги Выявлены отклонения и проблемы в клиентском сервисе, связанные с необоснованным отказом Банка в оказании услуги либо непредоставление услуги из-за недостатков'
'103': 'Нарушение срока оказания услуги\n Выявлены отклонения и проблемы в клиентском сервисе, связанные с нарушением срока оказания услуги (длительное ожидание клиентов в очереди, не предоставление услуг'
'104': 'Нарушение стандартов коммуникации с клиентами Выявлены отклонения и проблемы в клиентском сервисе, связанные с нарушениями стандартов коммуникации с клиентом (Внутренний стандарт по коммуникации)'
'105': 'Недобросовестные практики продаж Выявлены нарушения требований ФЗ и регуляторов в части применения недобросовестных практик продаж продуктов / услуг в физических и цифровых каналах, не отвечающих'
Прочитай диалог ниже и классифицируй его по заданным метрикам.

Правила:
1. Выбери только одну метрику
2. Выбери только номер метрики и больше ничего
3. Не пиши пояснений
4. Ответ должен содержать только число
5. Если ни одна из метрик абсолютно не подходит, выдавай None
"""

model_path = Path('/home/datalab/nfs/rag/llm_qwen_3_8b/Qwen_Qwen3-8B')
BATCH_SIZE = 1000
MAX_PROMPT_LEN = 16384

llm = (LLM(model=str(model_path), dtype='bfloat16', gpu_memory_utilization=0.9, max_model_len=16384, enable_prefix_caching=True,))
sampling_params = SamplingParams(temperature=0.1, max_tokens=10)

def process_batch(texts):
    prompts = [
        f'<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n'
        f'<|im_start|>user\n{text[:MAX_PROMPT_LEN]}<|im_end|>\n'
        f'<|im_start|>assistant\n'
        for text in texts
    ]
    outputs = llm.generate(prompts, sampling_params)
    batch_metrics = []
    
    for output in outputs:
        answer = output.outputs[0].text.strip()
        found = re.findall(r"\d+", answer)
        
        if found:
            batch_metrics.append(metric[0])
        else:
            batch_metrics.append(None)
    return batch_metrics

In [ ]:
import requests
import time
import os
import re

GIGACHAT_API_URL = "http://liveaccess/v1/gc/chat/completions"
HEADERS = {
    "Authorization": f"Bearer {os.environ.get('JPY_API_TOKEN')}",
    "Content-Type": "application/json"
}

SYSTEM_PROMPT = """
Ты профессионал по работе с клиентскими обращениями. Запомни метрики и их классификацию в рамках клиентского сервиса.
'101': 'Финансовые потери клиентов по вине Банка / компании Группы Выявлены отклонения и проблемы в клиентском сервисе, связанные с недостатками процессов Банка и повлекшие возникновение финансовых потерь'
'102': 'Необоснованный отказ в оказании услуги Выявлены отклонения и проблемы в клиентском сервисе, связанные с необоснованным отказом Банка в оказании услуги либо непредоставление услуги из-за недостатков'
'103': 'Нарушение срока оказания услуги\n Выявлены отклонения и проблемы в клиентском сервисе, связанные с нарушением срока оказания услуги (длительное ожидание клиентов в очереди, не предоставление услуг'
'104': 'Нарушение стандартов коммуникации с клиентами Выявлены отклонения и проблемы в клиентском сервисе, связанные с нарушениями стандартов коммуникации с клиентом (Внутренний стандарт по коммуникации)'
'105': 'Недобросовестные практики продаж Выявлены нарушения требований ФЗ и регуляторов в части применения недобросовестных практик продаж продуктов / услуг в физических и цифровых каналах, не отвечающих'
Прочитай диалог ниже и классифицируй его по заданным метрикам.

Правила:
1. Выбери только одну метрику
2. Выбери только номер метрики и больше ничего
3. Не пиши пояснений
4. Ответ должен содержать только число
"""

BATCH_SIZE = 1000
MAX_PROMPT_LEN = 16384

def ask_gigachat(messages: list) -> str:
    data = {
        "model": "GigaChat-3-Ultra",
        "messages": messages,
        "n": 1,
        "temperature": 0.01
    }
    attempt = 0
    while True:
        attempt += 1
        response = requests.post(GIGACHAT_API_URL, headers=HEADERS, json=data)
        if response.ok:
            return response.json()['choices'][0]['message']['content']
        elif not response.ok and attempt <= 20:
            time.sleep(1)
        else:
            status_code = response.status_code
            error_message = response.text
            if status_code == 400:
                raise RuntimeError(f"Ошибка в параметрах запроса: {error_message}")
            elif status_code == 401:
                raise RuntimeError(f"Авторизационный токен истек или не предоставлен: {error_message}")
            elif status_code == 404:
                raise RuntimeError(f"Не удалось найти модель: {error_message}")
            elif status_code == 405:
                raise RuntimeError(f"Ошибка при обработке введённого текста: {error_message}")
            elif status_code == 413:
                raise RuntimeError(f"Превышен максимальный размер входных данных: {error_message}")
            elif status_code == 500:
                raise RuntimeError(f"Внутренняя ошибка сервера: {error_message}")
            else:
                raise RuntimeError(f"Непредвиденная ошибка ({status_code}): {error_message}")

In [ ]:
metrics = []
for i in tqdm(range(0, len(documents), BATCH_SIZE)):
    batch_texts = documents[i: i + BATCH_SIZE]
    batch_metrics = process_batch(batch_texts)
    metrics.extend(batch_metrics)

print(f"Длины файлов: {len(metrics)} и {len(documents)}")

# дозаписывать в мету нормальную потом
# мету считать и перезаписать (потом встроить все это в prepare documents и save meta, чтобы убрать костыль)
path_to_save = '/home/datalab/nfs/rag/cache_le_finale2'
with open(f"{path_to_save}/metrics_meta.pkl", "wb") as f:
    pickle.dump({"metrics": metrics}, f)
    print("meta сохранена")

In [ ]:
# Сегодняшнее решение

import pickle

path_to_load = '/home/datalab/nfs/rag/cache_le_finale2'
def load_meta(path_to_load):
    global documents, doc_ids, tokenized_corpus, id_to_index, req_descs, msg_pprb_chats, req_reg_dates
    with open(f"{path_to_load}/meta.pkl", "rb") as f:
        meta = pickle.load(f)
    documents = meta["documents"]
    doc_ids = meta["doc_ids"]
    tokenized_corpus = meta["tokenized_corpus"]
    id_to_index = meta["id_to_index"]
    req_descs = meta["req_descs"]
    msg_pprb_chats = meta["msg_pprb_chats"]
    req_reg_dates = meta.get("req_reg_dates", [None] * len(documents))
    print("meta успешно загружена!")
    print(f"Количество документов: {len(documents)}")

load_meta()

In [ ]:
def prepare_for_metrics(kol):
    texts_needed = []
    number = 0
    for msg, short in zip(msg_pprb_chats[:kol], req_descs[:kol]):
        parts = []
        number += 1
        if short is not None and short.strip() != '':
            parts.append(f"Короткое описание обращения номер {number}: {short}")
        if msg is not None and msg.strip() != '':
            parts.append(f"Транскрибация обращения номер {number}: {msg}")
            
        combined_text = "\n".join(parts)
        texts_needed.append(combined_text)

In [ ]:
import json
import re

def batch_classify(texts: list, batch_size=8) -> list:
    results = []
    
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        
        # Формируем промпт (оставляем правила, чтобы модель их знала)
        batch_prompt = f"""
Ты профессионал по работе с клиентскими обращениями. Запомни метрики и их классификацию в рамках клиентского сервиса.
'101': 'Финансовые потери клиентов по вине Банка / компании Группы Выявлены отклонения и проблемы в клиентском сервисе, связанные с недостатками процессов Банка и повлекшие возникновение финансовых потерь'
'102': 'Необоснованный отказ в оказании услуги Выявлены отклонения и проблемы в клиентском сервисе, связанные с необоснованным отказом Банка в оказании услуги либо непредоставление услуги из-за недостатков'
'103': 'Нарушение срока оказания услуги\n Выявлены отклонения и проблемы в клиентском сервисе, связанные с нарушением срока оказания услуги (длительное ожидание клиентов в очереди, не предоставление услуг'
'104': 'Нарушение стандартов коммуникации с клиентами Выявлены отклонения и проблемы в клиентском сервисе, связанные с нарушениями стандартов коммуникации с клиентом (Внутренний стандарт по коммуникации)'
'105': 'Недобросовестные практики продаж Выявлены нарушения требований ФЗ и регуляторов в части применения недобросовестных практик продаж продуктов / услуг в физических и цифровых каналах, не отве

Классифицируй каждое обращение из списка ниже.
Правила:
1.Выбери только одну метрику
2.Выбери только номер метрики и больше ничего
3.Не пиши пояснений
4.Ответ должен содержать только число
5.Если ни одна из метрик абсолютно не подходит, выдавай None

Ответ должен быть строго в формате JSON (только объект с ключами-номерами).
# Пример того, как должен выглядеть твой ответ:
# Если я дам тебе 2 текста, верни: {{"1": "код", "2": "код"}}
# Если я дам тебе 3 текста, верни: {{"1": "код", "2": "код", "3": "код"}}
# Ключи (цифры) всегда начинаются с 1 для каждого нового списка текстов.

Тексты для классификации:
"""
        
        for idx, text in enumerate(batch, start=1):
            # Обрезаем текст для безопасности (лимит контекста)
            truncated_text = text[:4000] + '...' if len(text) > 4000 else text
            batch_prompt += f"\n--- ТЕКСТ {idx} ---\n{truncated_text}\n"
            
        messages = [{"role": "system", "content": batch_prompt}]
        
        response = ask_gigachat(messages)
        print(response)
        json_match = re.search(r'\{.*?\}', response)
        
        if json_match:
            # Достаем найденный фрагмент
            json_str = json_match.group(0)
            # Исправляем синтаксическую ошибку: Python None -> JSON null
            fixed_json_str = json_str.replace("None", "null")
            try:
                # Преобразуем исправленную строку в словарь Python
                response_json = json.loads(fixed_json_str)
                # Собираем результаты в нужном порядке (1, 2, 3...)
                # Используем .get(), чтобы избежать ошибки KeyError, если ключа нет
                batch_results = []
                for j in range(len(batch)):
                    # j+1 - это номер текста в текущем батче (1, 2...8)
                    # .get(str(j+1)) вернет значение или None, если ключа нет
                    batch_results.append(response_json.get(str(j+1)))
                # Добавляем обработанный батч в общие результаты
                results.extend(batch_results)
            except json.JSONDecodeError as e:
                # Если даже после замены None->null JSON сломан
                print(f"⚠️ Ошибка парсинга JSON: {e}")
                print("Текст, который вызвал ошибку:", fixed_json_str)
                # Чтобы программа не падала, добавляем пустые значения
                results.extend([None] * len(batch))
        else:
            # Если в ответе вообще не найдено фигурных скобок (нет JSON)
            print("⚠️ JSON не найден в ответе. Ответ был:")
            print(response)
            results.extend([None] * len(batch))
    return results # Возвращаем готовый список результатов